In [2]:
import pyrtl
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple

In [3]:
def reduce_or(wire):
    """Implement OR reduction for a wire vector"""
    result = wire[0]
    for i in range(1, len(wire)):
        result = result | wire[i]
    return result

def float_to_bf16(f: float) -> int:
    """Convert float to BF16 bit representation"""
    # cvt to float32 first
    f32_bits = np.float32(f).view(np.uint32)
    # keep top 16 bits
    bf16_bits = (f32_bits >> 16) & 0xFFFF
    return int(bf16_bits)

def bf16_to_float(bf16_bits: int) -> float:
    """Convert BF16 bits back to float"""
    # left shift to cvt to float32
    f32_bits = bf16_bits << 16
    # cvt to float32
    return float(np.uint32(f32_bits).view(np.float32))

In [43]:
class BF16Multiplier:
    def __init__(self):
        # inputs and output
        self.fp_a = pyrtl.Input(16, 'fp_a')
        self.fp_b = pyrtl.Input(16, 'fp_b')
        self.fp_out = pyrtl.Output(16, 'fp_out')
        
        # constants
        self.BIAS = pyrtl.Const(127, 8)
        self.INF_EXP = pyrtl.Const(0xFF, 8)
        self.ZERO_EXP = pyrtl.Const(0, 8)
        self.ZERO_MANT = pyrtl.Const(0, 7)
        
        # stage 1 -> 2
        self.reg_sign = pyrtl.Register(1, 'reg_sign')
        self.reg_mant1 = pyrtl.Register(8, 'reg_mant1')
        self.reg_mant2 = pyrtl.Register(8, 'reg_mant2')
        self.reg_exp1 = pyrtl.Register(8, 'reg_exp1')
        self.reg_exp2 = pyrtl.Register(8, 'reg_exp2')
        self.reg_special = pyrtl.Register(2, 'reg_special')
        self.reg_denorm1 = pyrtl.Register(1, 'reg_denorm1')
        self.reg_denorm2 = pyrtl.Register(1, 'reg_denorm2')
        
        # stage 2 -> 3
        self.reg_product = pyrtl.Register(16, 'reg_product')
        self.reg_exp_sum = pyrtl.Register(9, 'reg_exp_sum')
        self.reg_sign2 = pyrtl.Register(1, 'reg_sign2')
        self.reg_special2 = pyrtl.Register(2, 'reg_special2')
        
        # stage 3 -> 4
        self.reg_final_mant = pyrtl.Register(7, 'reg_final_mant')
        self.reg_final_exp = pyrtl.Register(8, 'reg_final_exp')
        self.reg_sign3 = pyrtl.Register(1, 'reg_sign3')
        self.reg_special3 = pyrtl.Register(2, 'reg_special3')
        
        self._build_pipeline()

    def stage1_extract(self):
        """stage 1: extract fields and detect special cases"""
        sign_a = self.fp_a[15]
        sign_b = self.fp_b[15]
        exp_a = self.fp_a[7:15]
        exp_b = self.fp_b[7:15]
        mant_a = self.fp_a[:7]
        mant_b = self.fp_b[:7]
        
        # detect special cases
        exp_a_all_ones = exp_a == self.INF_EXP
        exp_b_all_ones = exp_b == self.INF_EXP
        exp_a_zero = exp_a == self.ZERO_EXP
        exp_b_zero = exp_b == self.ZERO_EXP
        mant_a_zero = mant_a == self.ZERO_MANT
        mant_b_zero = mant_b == self.ZERO_MANT
        
        # detect denormals
        denorm_a = exp_a_zero & ~mant_a_zero
        denorm_b = exp_b_zero & ~mant_b_zero
        
        # special values detection
        inf_a = exp_a_all_ones & mant_a_zero
        inf_b = exp_b_all_ones & mant_b_zero
        nan_a = exp_a_all_ones & ~mant_a_zero
        nan_b = exp_b_all_ones & ~mant_b_zero
        zero_a = exp_a_zero & mant_a_zero
        zero_b = exp_b_zero & mant_b_zero
        
        # special cases
        is_nan = nan_a | nan_b | (inf_a & zero_b) | (zero_a & inf_b)
        is_inf = (inf_a & ~zero_b) | (inf_b & ~zero_a)
        is_zero = zero_a | zero_b
        
        special_val = pyrtl.select(is_nan, pyrtl.Const(2, 2),
                                 pyrtl.select(is_inf, pyrtl.Const(1, 2),
                                            pyrtl.select(is_zero, pyrtl.Const(3, 2),
                                                       pyrtl.Const(0, 2))))
        
        # handle mantissa for normal and denormal numbers
        mant1_hidden = pyrtl.select(exp_a_zero & ~denorm_a, 
                                  pyrtl.Const(0, 1),
                                  pyrtl.Const(1, 1))
        mant2_hidden = pyrtl.select(exp_b_zero & ~denorm_b,
                                  pyrtl.Const(0, 1),
                                  pyrtl.Const(1, 1))
        
        # register stage 1 outputs
        self.reg_sign.next <<= sign_a ^ sign_b
        self.reg_mant1.next <<= pyrtl.concat(mant1_hidden, mant_a)
        self.reg_mant2.next <<= pyrtl.concat(mant2_hidden, mant_b)
        self.reg_exp1.next <<= exp_a
        self.reg_exp2.next <<= exp_b
        self.reg_special.next <<= special_val
        self.reg_denorm1.next <<= denorm_a
        self.reg_denorm2.next <<= denorm_b

    def stage2_multiply(self):
        """stage 2: multiply mantissas and add exponents"""
        # mantissa multiplication
        product = self.reg_mant1 * self.reg_mant2
        
        # exponent addition
        exp1_adjusted = self.reg_exp1 + pyrtl.select(self.reg_denorm1, 
                                                    pyrtl.Const(1, 8), 
                                                    pyrtl.Const(0, 8))
        exp2_adjusted = self.reg_exp2 + pyrtl.select(self.reg_denorm2, 
                                                    pyrtl.Const(1, 8), 
                                                    pyrtl.Const(0, 8))
        
        exp_sum = pyrtl.concat(pyrtl.Const(0, 1), exp1_adjusted) + \
                 pyrtl.concat(pyrtl.Const(0, 1), exp2_adjusted) - \
                 pyrtl.concat(pyrtl.Const(0, 1), self.BIAS)
        
        # register stage 2 outputs
        self.reg_product.next <<= product
        self.reg_exp_sum.next <<= exp_sum
        self.reg_sign2.next <<= self.reg_sign
        self.reg_special2.next <<= self.reg_special

    def stage3_normalize(self):
        """stage 3: normalize and round result"""
        # get normalization position
        leading_one = self.reg_product[15]
        
        # select mantissa bits for rounding
        shifted_mant = pyrtl.select(leading_one,
                                  self.reg_product[8:15],
                                  self.reg_product[7:14])
        
        # rounding bits
        guard_bit = shifted_mant[1]
        round_bit = shifted_mant[0]
        sticky_bit = reduce_or(self.reg_product[:7])
        
        # round to nearest even
        round_up = (round_bit & (guard_bit | sticky_bit))
        
        # add rounding bit to mantissa
        mant_pre_round = shifted_mant[1:]
        rounded_mant = mant_pre_round + pyrtl.select(round_up,
                                                    pyrtl.Const(1, 7),
                                                    pyrtl.Const(0, 7))
        
        # check for mantissa overflow from rounding
        mant_overflow = (rounded_mant[6] & reduce_or(rounded_mant[:6])) == 0
        
        # adjust exponent based on normalization and rounding
        exp_adj = pyrtl.select(leading_one,
                             pyrtl.Const(1, 9),
                             pyrtl.Const(0, 9))
        exp_normalized = self.reg_exp_sum + exp_adj
        
        # final exponent adjustment for mantissa overflow
        final_exp = pyrtl.select(mant_overflow,
                               exp_normalized + 1,
                               exp_normalized)
        
        # handle overflow/underflow
        is_overflow = final_exp >= pyrtl.concat(pyrtl.Const(0, 1), self.INF_EXP)
        is_underflow = final_exp <= pyrtl.concat(pyrtl.Const(0, 1), self.ZERO_EXP)
        
        # select final mantissa
        final_mant = pyrtl.select(mant_overflow,
                                pyrtl.Const(0, 7),
                                rounded_mant)
        
        # select final exponent
        exp_out = pyrtl.select(is_overflow, self.INF_EXP,
                             pyrtl.select(is_underflow, self.ZERO_EXP,
                                        final_exp[0:8]))
        
        # register stage 3 outputs
        self.reg_final_mant.next <<= final_mant
        self.reg_final_exp.next <<= exp_out
        self.reg_sign3.next <<= self.reg_sign2
        self.reg_special3.next <<= self.reg_special2

    def stage4_output(self):
        """stage 4: format final output"""
        # check special cases
        is_normal = self.reg_special3 == pyrtl.Const(0, 2)
        is_inf = self.reg_special3 == pyrtl.Const(1, 2)
        is_nan = self.reg_special3 == pyrtl.Const(2, 2)
        
        # construct output formats
        normal_result = pyrtl.concat_list([
            self.reg_sign3,
            self.reg_final_exp,
            self.reg_final_mant
        ])
        
        inf_result = pyrtl.concat_list([
            self.reg_sign3,
            self.INF_EXP,
            self.ZERO_MANT
        ])
        
        nan_result = pyrtl.concat_list([
            pyrtl.Const(0, 1),
            self.INF_EXP,
            pyrtl.Const(1, 7)
        ])
        
        zero_result = pyrtl.concat_list([
            self.reg_sign3,
            self.ZERO_EXP,
            self.ZERO_MANT
        ])
        
        # select final output
        self.fp_out <<= pyrtl.select(is_nan, nan_result,
                                   pyrtl.select(is_inf, inf_result,
                                              pyrtl.select(is_normal, normal_result,
                                                         zero_result)))

    def _build_pipeline(self):
        """connect all pipeline stages"""
        self.stage1_extract()
        self.stage2_multiply()
        self.stage3_normalize()
        self.stage4_output()

In [44]:
def create_test_vectors() -> Dict[str, List[Tuple[float, float, float]]]:
    """Create comprehensive test vectors"""
    return {
        "basic": [
            # a, b, expected
            (1.0, 1.0, 1.0),
            (2.0, 3.0, 6.0),
            (4.0, 0.5, 2.0),
            (-2.0, 3.0, -6.0),
            (-2.0, -3.0, 6.0),
            (0.1, 0.1, 0.01),
        ],
        "subnormal": [
            (1.0, 1.1754944e-38, 1.1754944e-38),  # min normal
            (2.0, 5.877472e-39, 1.1754944e-38),   # min subnormal * 2
            (1.0, 5.877472e-39, 5.877472e-39),    # min subnormal
        ],
        "rounding": [
            (1.0, 1.0 + 2**(-8), 1.0),      # round down
            (1.0, 1.0 + 2**(-7), 1.0625),   # round up
            (1.0, 1.0 - 2**(-7), 0.9375),   # negative round
        ],
        "special_values": [
            (float('inf'), 1.0, float('inf')),     # infinity
            (float('inf'), -1.0, float('-inf')),   # negative infinity
            (float('inf'), 0.0, float('nan')),     # inf * 0
            (float('nan'), 1.0, float('nan')),     # nan propagation
            (0.0, 0.0, 0.0),                       # zero
            (-0.0, 1.0, -0.0),                     # signed zero
        ],
        "overflow_underflow": [
            (3.38953139e38, 2.0, float('inf')),    # overflow to inf
            (1.1754944e-38, 1e-2, 0.0),           # underflow to zero
            (3.38953139e38, 1e-40, 1.0),          # large * small = normal
        ],
        "exactness": [
            (2.0, 0.5, 1.0),                # exact result
            (3.0, 1.0/3.0, 1.0),            # inexact result
            (1.1, 1.1, 1.2100000381469727), # rounding required
        ]
    }

def run_bf16_test(test_vectors: Dict[str, List[Tuple[float, float, float]]]) -> pd.DataFrame:
    """Run test vectors through the BF16 multiplier"""
    results = []
    
    for category, vectors in test_vectors.items():
        for a, b, expected in vectors:
            # cvt inputs to bf16
            a_bits = float_to_bf16(a)
            b_bits = float_to_bf16(b)
            expected_bits = float_to_bf16(expected)
            
            # set up simulation
            pyrtl.reset_working_block()
            multiplier = BF16Multiplier()
            sim = pyrtl.Simulation()
            
            # run for 5 cycles to handle pipeline
            for _ in range(5):
                sim.step({'fp_a': a_bits, 'fp_b': b_bits})
            
            # get result and convert back to float
            result_bits = sim.inspect('fp_out')
            result = bf16_to_float(result_bits)
            
            # check if result matches expected
            # special handling for nan comparison
            if np.isnan(expected):
                passed = np.isnan(result)
            else:
                # for normal numbers, check relative error
                if expected != 0:
                    rel_error = abs((result - expected) / expected)
                    passed = rel_error < 1e-4  # Allow small relative error
                else:
                    # for zero, check absolute error
                    passed = abs(result) < 1e-45  # should be exactly zero
            
            results.append({
                'category': category,
                'input_a': a,
                'input_b': b,
                'expected': expected,
                'result': result,
                'passed': passed
            })
    
    return pd.DataFrame(results)

def analyze_test_results(df: pd.DataFrame) -> None:
    """Analyze and display test results"""
    print("\nTest Results Summary")
    print("-" * 50)
    print(f"Total test cases: {len(df)}")
    print(f"Passed: {df['passed'].sum()}")
    print(f"Failed: {(~df['passed']).sum()}")
    
    print("\nResults by Category")
    print("-" * 50)
    category_results = df.groupby('category').agg({
        'passed': ['count', 'sum', lambda x: f"{(sum(x)/len(x))*100:.1f}%"]
    })
    category_results.columns = ['Total', 'Passed', 'Pass Rate']
    print(category_results)
    
    # failed cases
    failed = df[~df['passed']]
    if len(failed) > 0:
        print("\nFailed Cases")
        print("-" * 50)
        for _, case in failed.iterrows():
            print(f"Category: {case['category']}")
            print(f"Input A: {case['input_a']}")
            print(f"Input B: {case['input_b']}")
            print(f"Expected: {case['expected']}")
            print(f"Got: {case['result']}")
            print()


Test Results Summary
--------------------------------------------------
Total test cases: 24
Passed: 1
Failed: 23

Results by Category
--------------------------------------------------
                    Total  Passed Pass Rate
category                                   
basic                   6       0      0.0%
exactness               3       0      0.0%
overflow_underflow      3       0      0.0%
rounding                3       0      0.0%
special_values          6       1     16.7%
subnormal               3       0      0.0%

Failed Cases
--------------------------------------------------
Category: basic
Input A: 1.0
Input B: 1.0
Expected: 1.0
Got: 2.350988701644575e-38

Category: basic
Input A: 2.0
Input B: 3.0
Expected: 6.0
Got: 2.424457098570968e-38

Category: basic
Input A: 4.0
Input B: 0.5
Expected: 2.0
Got: 2.3877229001077715e-38

Category: basic
Input A: -2.0
Input B: 3.0
Expected: -6.0
Got: 2.442824197802566e-38

Category: basic
Input A: -2.0
Input B: -3.0
Expected: 6.0

In [ ]:
test_vectors = create_test_vectors()
results_df = run_bf16_test(test_vectors)
analyze_test_results(results_df)